In [2]:
import pandas as pd
import numpy as np, json, re
from sklearn.model_selection import cross_val_score, KFold
from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier
from sklearn.metrics import make_scorer, mean_squared_error

In [3]:
# --- 1. Load the dataset ---
df = pd.read_csv('../../../data/test-final/FINAL_master.csv')

In [4]:
# 2. Parse embeddings
def parse_embed(s):
    try:
        return np.array(json.loads(s))
    except:
        parts = re.split(r'[,\s]+', s.strip().lstrip('[').rstrip(']'))
        return np.array([float(x) for x in parts if x])
df['embed_vec'] = df['transcript_embedding'].apply(parse_embed)
df = df[df['embed_vec'].map(lambda a: a.size>0)]

In [5]:
# 3a. Regression target
y_reg = df['Overall_Score'].values

In [6]:
# 3b. Classification target
rec_cols = ['rec_Hire','rec_Consider','rec_Reject']
y_cls = df[rec_cols].values.argmax(axis=1)

In [7]:
# 4. Feature matrix
X = np.vstack(df['embed_vec'].values)

In [8]:
# 5. CV setup
cv = KFold(n_splits=5, shuffle=True, random_state=42)

In [9]:
# 6a. Regression CV
reg = GradientBoostingRegressor(random_state=42)
mse_scorer = make_scorer(mean_squared_error, greater_is_better=False)
mse_scores = cross_val_score(reg, X, y_reg, cv=cv, scoring=mse_scorer)
rmse_scores = np.sqrt(-mse_scores)
print("Text‑Only CV RMSE:", np.round(rmse_scores,3))
print("Mean RMSE:", np.round(rmse_scores.mean(),3))

Text‑Only CV RMSE: [0.106 0.099 0.106 0.088 0.089]
Mean RMSE: 0.098


In [10]:
# 6b. Classification CV
clf = RandomForestClassifier(random_state=42)
acc_scores = cross_val_score(clf, X, y_cls, cv=cv, scoring='accuracy')
print("Text‑Only CV Accuracy:", np.round(acc_scores,3))
print("Mean Accuracy:", np.round(acc_scores.mean(),3))

Text‑Only CV Accuracy: [0.705 0.65  0.767 0.767 0.717]
Mean Accuracy: 0.721
